### Modeling phase

### Objective

#### Imports

In [146]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


#remove outliers (maybe)
#check imbalance (smote or undersample if needed)
#imputation method for crp_normalized
#imputation method for esr
#groupkfold (number of splits)


#### Load preprocessed data

In [147]:
#Load the dataset
df_cleaned = pd.read_csv('../data/RA_dataset_cleaned.csv')

df_cleaned.head()

,patient_id,gender,year_of_diagnosis,rf_status,anti_ccp_status,comorbidities,visit_date,visit_type,crp,crp_upper_limit,...,comorb_gastrointestinal_hepatic,comorb_malignancy,comorb_musculoskeletal,comorb_neurologic_psychiatric,comorb_autoimmune_other,comorbidity_count,csdmard_use,b_ts_dmard_use,steroid_use,treatment_count
0,RA001,1,1997.0,0.0,0.0,"depression,hypothyroidism",2017-12-18,Baseline,0.40,0.5,...,0,0,0,1,0,2,1,1,0,3
1,RA001,1,1997.0,0.0,0.0,"depression,hypothyroidism",2018-03-20,Follow-up,NaN,NaN,...,0,0,0,1,0,2,1,0,0,2
2,RA001,1,1997.0,0.0,0.0,"depression,hypothyroidism",2018-09-25,Follow-up,0.40,0.5,...,0,0,0,1,0,2,0,1,0,1
3,RA002,1,1999.0,0.0,0.0,NaN,2023-10-27,Baseline,0.02,0.5,...,0,0,0,0,0,0,1,1,0,3
4,RA002,1,1999.0,0.0,0.0,NaN,2024-01-30,Follow-up,0.07,0.5,...,0,0,0,0,0,0,1,1,0,2


#### Initial checks of the modeling dataset

In [148]:
df_cleaned.shape

(228, 46)

In [149]:
df_cleaned['target_flare_next'].value_counts()

target_flare_next
0    138
1     90
Name: count, dtype: int64

#### Define target, groups, and candidate features

In [150]:
#Define the target variable
target_variable = 'target_flare_next'

#Drop the columns that are not needed for modeling plus the target variable,
# after EDA analysis.

drop_columns = [
    target_variable,
    'patient_id',
    'visit_date', 
    'source',
    'year_of_diagnosis',
    'crp_upper_limit',
    'comorbidities',
    'visit_type',
    'csdmard_name',
    'b_ts_dmard_name', 
    'treatment_decision', 
    'switch_reason'
    
]

#Define the feature matrix X 
X = df_cleaned.drop(columns = drop_columns, errors='ignore')

#Define the target variable y
y = df_cleaned[target_variable]

#Define the groups for group-aware splitting
groups = df_cleaned['patient_id']

In [151]:
print('x shape: ', X.shape)
print('y shape: ', y.shape)
print("groups shape:", groups.shape)

x shape:  (228, 34)
y shape:  (228,)
groups shape: (228,)


In [152]:
continuous_features = [
    'age_at_visit',
    'disease_duration',
    'das28_score',
    'das28_change',
    'esr',
    'crp_normalized', 
    
    'mtx_use'
]


binary_features=[
    'gender', 
    'rf_status',
    'anti_ccp_status'


]

#### Build preprocessing pipeline

In [153]:
continuous_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

binary_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='if_binary'))
])

preprocessor = ColumnTransformer(transformers=[
    ('cont', continuous_transformer, continuous_features),
    ('bin', binary_transformer, binary_features)
])

preprocessor



,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cont', ...), ('bin', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. 

In [154]:
group_kfold = GroupKFold(n_splits=2, shuffle=True, random_state=42)
for i, (train_index, test_index) in enumerate(group_kfold.split(X, y, groups)):
    
    print(f"  Train: index={train_index}, group={groups.iloc[train_index].values}")
    print(f"  Test:  index={test_index}, group={groups.iloc[test_index].values}")
    print(f"Fold {i}, overlap:", set(groups.iloc[train_index]).intersection(groups.iloc[test_index]))

  Train: index=[  3   4   5   6   7   8   9  16  17  18  19  20  21  23  24  25  34  35
  36  37  38  44  45  46  47  48  49  50  52  53  63  64  65  66  67  68
  69  74  75  76  78  79  80  83  84  85  86  87  88  89  90  99 100 101
 109 110 111 112 119 120 121 122 123 132 133 134 135 154 155 156 157 160
 161 162 163 164 181 182 183 197 198 199 200 201 202 203 204 205 206 209
 210 211 212 213 214 215 216 217 220 221 222 223 224 225 226 227], group=<StringArray>
['RA002', 'RA002', 'RA002', 'RA002', 'RA003', 'RA004', 'RA004', 'RA007',
 'RA007', 'RA007',
 ...
 'RA067', 'RA067', 'RA069', 'RA070', 'RA070', 'RA070', 'RA070', 'RA070',
 'RA070', 'RA070']
Length: 106, dtype: str
  Test:  index=[  0   1   2  10  11  12  13  14  15  22  26  27  28  29  30  31  32  33
  39  40  41  42  43  51  54  55  56  57  58  59  60  61  62  70  71  72
  73  77  81  82  91  92  93  94  95  96  97  98 102 103 104 105 106 107
 108 113 114 115 116 117 118 124 125 126 127 128 129 130 131 136 137 138
 139 140 141 

#### Logistic Regression


In [155]:

model_lr = LogisticRegression(
    max_iter = 1000, 
    penalty = 'l2',
    solver = 'lbfgs',
    class_weight = 'balanced', 
    random_state = 42)

model_lr.fit(X_train, y_train)
y_pred = model_lr.predict(X_test)
print(classification_report(y_test, y_pred))
print('Accuracy: ', accuracy_score(y_test, y_pred))

c:\Users\konna\Thesis_Project\ra_env\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

logistic regression  
penalty=l2 (ridge)  
C: the strength of regularization  
solver: liblinear, lbfgs  
max_iter = 1000  
class_weight='balanced' (if my dataset is imbalanced)  
random state: usually 42  
fit(X,y)
predict(X,y)
predict_proba(X)
score(X,y)
decision_function(X) (optional)
sample_weight (optional)


#### Random Forest

In [156]:
model_rf = RandomForestClassifier(
        n_estimators=300,
        max_depth = 3,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42, 
        n_jobs=-1
)

model_pipeline = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', model_rf)
])

y_pred_rf = cross_val_predict(model_pipeline,
                            X,
                            y,
                            cv=group_kfold,
                            groups=groups,
                            method='predict')


y_prob_rf = cross_val_predict(model_pipeline,
                            X,
                            y,
                            cv=group_kfold,
                            groups=groups,
                            method='predict_proba')[:, 1]

print(classification_report(y, y_pred_rf))
print(f"Accuracy: {accuracy_score(y, y_pred_rf):.2f}")


              precision    recall  f1-score   support

           0       0.65      0.71      0.68       138
           1       0.49      0.42      0.45        90

    accuracy                           0.60       228
   macro avg       0.57      0.57      0.57       228
weighted avg       0.59      0.60      0.59       228

Accuracy: 0.60


#### XGBoost

#### CatBoost